# 01 - dUCk Data Preparation

This notebook prepares the official English-Malay training file, creates a stratified validation split, and keeps the official testing file separate.

In [ ]:
from pathlib import Path
from html import unescape
import random
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / 'eng_malay_training_dataset.csv'
TEST_PATH = RAW_DIR / 'eng_malay_testing_dataset.csv'

assert TRAIN_PATH.exists(), f'Missing file: {TRAIN_PATH}'
assert TEST_PATH.exists(), f'Missing file: {TEST_PATH}'

In [ ]:
official_train = pd.read_csv(TRAIN_PATH)
official_test = pd.read_csv(TEST_PATH)

print('Official training shape:', official_train.shape)
print('Official testing shape:', official_test.shape)
print('Training columns:', official_train.columns.tolist())
print('Testing columns:', official_test.columns.tolist())
display(official_train.head())

In [ ]:
def resolve_column(frame, candidates):
    lookup = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate in lookup:
            return lookup[candidate]
    raise ValueError(f'Could not find any of {candidates}. Available columns: {frame.columns.tolist()}')

URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
USER_PATTERN = re.compile(r'(?<!\w)@\w+')

def clean_xlmr_text(value):
    # Keep case, punctuation, emoji, slang, hashtags, and English-Malay mixing.
    text = unescape(str(value))
    text = URL_PATTERN.sub(' <URL> ', text)
    text = USER_PATTERN.sub(' <USER> ', text)
    return re.sub(r'\s+', ' ', text).strip()

def prepare_frame(frame):
    frame = frame.copy()
    frame = frame.loc[:, ~frame.columns.astype(str).str.lower().str.startswith('unnamed')]
    text_column = resolve_column(frame, ['tweettext', 'tweet', 'text'])
    label_column = resolve_column(frame, ['tweetsentiment', 'sentiment', 'label'])
    frame = frame[[text_column, label_column]].rename(
        columns={text_column: 'tweet_text_raw', label_column: 'original_label'}
    )
    frame['tweet_text'] = frame['tweet_text_raw'].map(clean_xlmr_text)
    frame['original_label'] = pd.to_numeric(frame['original_label'], errors='raise').astype(int)
    label_mapping = {-1: 0, 0: 1, 1: 2}
    unexpected = set(frame['original_label'].unique()) - set(label_mapping)
    if unexpected:
        raise ValueError(f'Unexpected labels: {sorted(unexpected)}')
    frame['label'] = frame['original_label'].map(label_mapping).astype(int)
    if frame['tweet_text'].isna().any() or frame['tweet_text'].eq('').any():
        raise ValueError('Empty tweet text detected.')
    return frame.reset_index(drop=True)

prepared_train = prepare_frame(official_train)
prepared_test = prepare_frame(official_test)

# Remove semantic duplicates from training after URL/user normalisation.
# The official test set itself remains unchanged and held out.
test_text_keys = set(prepared_test['tweet_text'].str.casefold())
overlap_mask = prepared_train['tweet_text'].str.casefold().isin(test_text_keys)
print('Training rows removed due to test overlap:', int(overlap_mask.sum()))
prepared_train = prepared_train.loc[~overlap_mask].reset_index(drop=True)

print('Training duplicates:', prepared_train['tweet_text'].str.casefold().duplicated().sum())
print('Testing duplicates:', prepared_test['tweet_text'].str.casefold().duplicated().sum())
print('Cross-file overlap:', len(set(prepared_train['tweet_text'].str.casefold()) & set(prepared_test['tweet_text'].str.casefold())))
print('Training labels:\n', prepared_train['original_label'].value_counts().sort_index())
print('Testing labels:\n', prepared_test['original_label'].value_counts().sort_index())

In [ ]:
train_data, validation_data = train_test_split(
    prepared_train,
    test_size=0.20,
    random_state=SEED,
    stratify=prepared_train['label'],
)

train_data = train_data.reset_index(drop=True)
validation_data = validation_data.reset_index(drop=True)

train_data.to_csv(PROCESSED_DIR / 'train.csv', index=False)
validation_data.to_csv(PROCESSED_DIR / 'validation.csv', index=False)
prepared_test.to_csv(PROCESSED_DIR / 'test.csv', index=False)

for name, frame in [('train', train_data), ('validation', validation_data), ('held-out test', prepared_test)]:
    print(f'\n{name}: {len(frame)} rows')
    print(frame['original_label'].value_counts(normalize=True).sort_index().round(3))

print('\nSaved processed files to:', PROCESSED_DIR)